# 03 — Traditional Volatility Models 

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "results" / "tables"
DIAG = ROOT / "results" / "diagnostics"

for p in [TABLES, DIAG]:
    p.mkdir(parents=True, exist_ok=True)

HORIZONS = [5, 21, 63]
VAL = pd.Timestamp("2023-01-01")
TEST = pd.Timestamp("2025-01-01")
EPS = 1e-10

def qlike(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    ratio = np.maximum(y**2, EPS) / np.maximum(p**2, EPS)
    return float(np.mean(ratio - np.log(ratio) - 1))

def metrics(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    e = p - y
    return {
        "N": len(y),
        "MAE": float(np.mean(np.abs(e))),
        "RMSE": float(np.sqrt(np.mean(e**2))),
        "QLIKE": qlike(y, p),
        "Correlation": float(np.corrcoef(y, p)[0, 1]) if np.std(p) > 0 else np.nan,
        "Bias": float(np.mean(e)),
        "Actual_Std": float(np.std(y, ddof=1)) if len(y) > 1 else np.nan,
        "Prediction_Std": float(np.std(p, ddof=1)) if len(p) > 1 else np.nan,
    }

## 1. Finite-value input audit

In [2]:
audit_rows = []

for h in HORIZONS:
    d = pd.read_parquet(PROCESSED / f"volatility_features_h{h}.parquet")
    for c in ["Sector_Return", "RV_5", "RV_21", "RV_63", f"Target_RV_{h}"]:
        v = pd.to_numeric(d[c], errors="coerce")
        audit_rows.append({
            "Horizon": h,
            "Variable": c,
            "NaN": int(v.isna().sum()),
            "Positive_Infinity": int(np.isposinf(v).sum()),
            "Negative_Infinity": int(np.isneginf(v).sum()),
            "Maximum_Finite": v[np.isfinite(v)].max() if np.isfinite(v).any() else np.nan
        })

finite_audit = pd.DataFrame(audit_rows)
finite_audit.to_csv(DIAG / "03_input_finite_value_audit.csv", index=False)
display(finite_audit)

if finite_audit[["Positive_Infinity", "Negative_Infinity"]].to_numpy().sum() > 0:
    print("Warning: non-finite values will be treated as missing below.")

,Horizon,Variable,NaN,Positive_Infinity,Negative_Infinity,Maximum_Finite
0,5,Sector_Return,0,0,0,0.212396
1,5,RV_5,4,0,0,2.793305
2,5,RV_21,20,0,0,2.083992
3,5,RV_63,62,0,0,1.797901
4,5,Target_RV_5,0,0,0,2.793305
5,21,Sector_Return,0,0,0,0.212396
6,21,RV_5,4,0,0,2.793305
7,21,RV_21,20,0,0,2.083992
8,21,RV_63,62,0,0,1.797901
9,21,Target_RV_21,0,0,0,2.083992


## 2. Generate traditional forecasts

In [3]:
pred = []

for h in HORIZONS:
    d = (
        pd.read_parquet(PROCESSED / f"volatility_features_h{h}.parquet")
        .sort_values("Date")
        .reset_index(drop=True)
    )
    d["Date"] = pd.to_datetime(d["Date"])

    numeric_cols = d.select_dtypes(include=[np.number]).columns
    d[numeric_cols] = d[numeric_cols].replace([np.inf, -np.inf], np.nan)

    target = f"Target_RV_{h}"

    # ----------------------------------------------------------
    # Static baselines
    # ----------------------------------------------------------
    static_models = [
        ("Persistence_21D", "RV_21"),
        ("Historical_Matched", f"RV_{h}"),
        ("EWMA_094", "EWMA_Vol_094"),
    ]

    for _, row in d.iterrows():
        if row["Date"] < VAL or not np.isfinite(row[target]):
            continue

        period = "Validation" if row["Date"] < TEST else "Test"

        for model_name, feature_name in static_models:
            if feature_name not in d.columns:
                continue
            p = row[feature_name]
            if np.isfinite(p) and p > 0:
                pred.append([
                    row["Date"], h, period, model_name,
                    float(row[target]), float(p)
                ])

    # ----------------------------------------------------------
    # Expanding HAR-RV with h-row target embargo
    # ----------------------------------------------------------
    har_features = ["RV_5", "RV_21", "RV_63"]

    for i, row in d.iterrows():
        if row["Date"] < VAL or i - h < 500:
            continue
        if not np.isfinite(row[target]):
            continue

        x_now = pd.to_numeric(row[har_features], errors="coerce").to_numpy(dtype=float)
        if not np.isfinite(x_now).all():
            continue

        tr = (
            d.iloc[:i-h]
            .dropna(subset=har_features + [target])
            .copy()
        )

        if len(tr) < 500:
            continue

        model = LinearRegression().fit(tr[har_features], tr[target])
        p = float(model.predict(row[har_features].to_frame().T)[0])
        p = max(p, 1e-8)

        pred.append([
            row["Date"], h,
            "Validation" if row["Date"] < TEST else "Test",
            "HAR_RV", float(row[target]), p
        ])

    # ----------------------------------------------------------
    # True expanding GARCH(1,1)
    # ----------------------------------------------------------
    try:
        from arch import arch_model

        for i, row in d.iterrows():
            if row["Date"] < VAL or i < 500 or not np.isfinite(row[target]):
                continue

            hist = (
                d.loc[:i, "Sector_Return"]
                .replace([np.inf, -np.inf], np.nan)
                .dropna()
                .astype(float)
                .to_numpy()
            )

            if len(hist) < 500:
                continue

            try:
                hist_pct = hist * 100.0
                fit = arch_model(
                    hist_pct,
                    mean="Zero",
                    vol="GARCH",
                    p=1,
                    q=1,
                    dist="normal",
                    rescale=False
                ).fit(disp="off", show_warning=False)

                var = fit.forecast(horizon=h, reindex=False).variance.values[-1]
                p = np.sqrt(252 * np.mean(var)) / 100.0

                if np.isfinite(p) and p > 0:
                    pred.append([
                        row["Date"], h,
                        "Validation" if row["Date"] < TEST else "Test",
                        "GARCH_11", float(row[target]), float(p)
                    ])

            except Exception:
                continue

    except ImportError:
        print("Package 'arch' is not installed. Install it with: pip install arch")

pred = pd.DataFrame(
    pred,
    columns=["Date", "Horizon", "Period", "Model", "Actual", "Prediction"]
)

pred = (
    pred.replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["Actual", "Prediction"])
    .query("Actual >= 0 and Prediction > 0")
    .sort_values(["Horizon", "Period", "Date", "Model"])
    .reset_index(drop=True)
)

pred.to_csv(TABLES / "03_traditional_common_sample_predictions.csv", index=False)

counts = (
    pred.groupby(["Horizon", "Period", "Model"])
    .size()
    .rename("N")
    .reset_index()
)
counts.to_csv(DIAG / "03_prediction_counts.csv", index=False)
display(counts)

,Horizon,Period,Model,N
0,5,Test,EWMA_094,430
1,5,Test,GARCH_11,430
2,5,Test,HAR_RV,430
3,5,Test,Historical_Matched,430
4,5,Test,Persistence_21D,430
5,5,Validation,EWMA_094,506
6,5,Validation,GARCH_11,506
7,5,Validation,HAR_RV,506
8,5,Validation,Historical_Matched,506
9,5,Validation,Persistence_21D,506


## 3. Metrics and common-date diagnostics

In [4]:
metric_rows = []

for keys, x in pred.groupby(["Horizon", "Period", "Model"]):
    metric_rows.append({
        "Horizon": keys[0],
        "Period": keys[1],
        "Model": keys[2],
        **metrics(x["Actual"], x["Prediction"])
    })

traditional_metrics = pd.DataFrame(metric_rows).sort_values(
    ["Horizon", "Period", "QLIKE"]
)

traditional_metrics.to_csv(TABLES / "03_traditional_metrics.csv", index=False)
display(traditional_metrics)

common_rows = []
for (h, period), x in pred.groupby(["Horizon", "Period"]):
    date_sets = [set(g["Date"]) for _, g in x.groupby("Model")]
    common = set.intersection(*date_sets) if date_sets else set()
    common_rows.append({
        "Horizon": h,
        "Period": period,
        "Models": x["Model"].nunique(),
        "Common_Dates": len(common),
        "First_Common_Date": min(common) if common else pd.NaT,
        "Last_Common_Date": max(common) if common else pd.NaT,
    })

common = pd.DataFrame(common_rows)
common.to_csv(DIAG / "03_common_date_audit.csv", index=False)
display(common)

,Horizon,Period,Model,N,MAE,RMSE,QLIKE,Correlation,Bias,Actual_Std,Prediction_Std
2,5,Test,HAR_RV,430,0.077804,0.102149,0.348300,0.416501,0.009373,0.109326,0.067714
1,5,Test,GARCH_11,430,0.084210,0.109472,0.349235,0.397021,0.031767,0.109326,0.073938
0,5,Test,EWMA_094,430,0.079599,0.105857,0.384502,0.372346,0.011833,0.109326,0.068926
4,5,Test,Persistence_21D,430,0.082476,0.112235,0.431247,0.332479,0.009820,0.109326,0.079919
3,5,Test,Historical_Matched,430,0.085580,0.114800,0.609748,0.449485,-0.001701,0.109326,0.109716
5,5,Validation,EWMA_094,506,0.052508,0.064759,0.281983,0.057064,0.010354,0.059880,0.026235
7,5,Validation,HAR_RV,506,0.057560,0.070194,0.297451,0.045973,0.024557,0.059880,0.030226
6,5,Validation,GARCH_11,506,0.064612,0.077870,0.315471,0.061059,0.041806,0.059880,0.031085
9,5,Validation,Persistence_21D,506,0.053711,0.066180,0.319046,0.056945,0.007849,0.059880,0.030848
8,5,Validation,Historical_Matched,506,0.065903,0.084298,0.900727,0.009224,0.000588,0.059880,0.060004


,Horizon,Period,Models,Common_Dates,First_Common_Date,Last_Common_Date
0,5,Test,5,430,2025-01-02,2026-09-11
1,5,Validation,5,506,2023-01-03,2024-12-31
2,21,Test,5,414,2025-01-02,2026-08-20
3,21,Validation,5,506,2023-01-03,2024-12-31
4,63,Test,5,372,2025-01-02,2026-06-23
5,63,Validation,5,506,2023-01-03,2024-12-31
